# Tutorial 10: Does Word Position Help? IBM Model 2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/tutorials/10-ibm-model-2.ipynb)

**Reading. About 25 minutes; it runs in seconds.**

IBM Model 1, which Lecture 18's notebook ran, learns which word translates which from counts
alone, and never looks at where a word sits in its sentence. IBM Model 2 (Brown and colleagues,
1993) adds one thing: a table of how likely the word at one position is to translate the word at
another. This notebook runs both on fifteen English-French sentence pairs and asks when knowing
the position helps: where the two languages keep the same word order, and where French puts the
adjective after the noun.

By the end you will be able to:

1. say what Model 2 adds to Model 1, and how the same EM learns its table of positions alongside
   its table of words;
2. explain why word position settles an alignment where two languages keep the same order, and
   misleads where they reorder;
3. explain how attention uses word position without a table of positions, so that the words
   themselves can decide.

These serve Lecture 18's outcome *Explain how word alignments relate to attention in NMT*.

One prediction stops the notebook until you write it; otherwise you run the cells and read.

**Running in Google Colab?** Run the next two cells: the first installs TorchLingo, the second
checks the setup. There is nothing to uncomment, nothing to download, and no GPU is needed.


<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**TorchLingo tutorial** — assigned reading for Lecture 18 (*Open Problems, and What Earlier MT Methods Teach*). A head start on the **Final Project**.

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo


In [ ]:
# Check the setup: the TorchLingo and PyTorch versions and the device, printed here so that
# anything wrong shows up now rather than ten cells from now.
from torchlingo.colab import setup

env = setup()

from collections import defaultdict

import torch

from torchlingo.visualization import format_attention


## Step 1: fifteen sentence pairs

The pairs below are the whole corpus. You need no French to follow them: each French sentence
says its English word for word, in the same order, with one exception. French puts most
adjectives after the noun (*le chat noir*, the cat black), and a few short ones before it, as
English does (*le grand chien*, the big dog).

Two pairs are there to be hard. In *the young fox sees the cat* and *the grey owl sees the cat*,
the adjective and the noun appear nowhere else in the corpus, in either language. Counting which
words turn up together cannot tell `jeune` (young) from `renard` (fox), or `chouette` (owl) from
`grise` (grey). The fox keeps English order, *le jeune renard*; the owl does not, *la chouette
grise*.


In [ ]:
PAIRS = [
    ("the cat sleeps", "le chat dort"),
    ("the dog sleeps", "le chien dort"),
    ("the woman sleeps", "la femme dort"),
    ("a dog runs", "un chien court"),
    ("the cat runs", "le chat court"),
    ("the cat sees the dog", "le chat voit le chien"),
    ("the dog sees the woman", "le chien voit la femme"),
    ("the woman sees a cat", "la femme voit un chat"),
    ("the black cat sleeps", "le chat noir dort"),
    ("a white dog runs", "un chien blanc court"),
    ("the cat sees the big dog", "le chat voit le grand chien"),
    ("the woman sees the small cat", "la femme voit le petit chat"),
    ("the dog sees the small woman", "le chien voit la petite femme"),
    ("the young fox sees the cat", "le jeune renard voit le chat"),
    ("the grey owl sees the cat", "la chouette grise voit le chat"),
]

# Each pair as two lists of words.
pairs = [(english.split(), french.split()) for english, french in PAIRS]
print(f"{len(pairs)} pairs")


## Step 2: Model 1, which ignores position

Model 1 keeps one number for each pair of words, t(f | e): the probability that English word
`e` translates as French word `f`. Lecture 18's notebook wrote its EM out step by step; here it
is again, condensed. Each round shares every French word among its pair's English words in
proportion to t (expect), then adds the shares up over the corpus into a new t (maximize).

`alignment` reads one pair's alignment off the tables: for each French word, its share across
that pair's English words, each row summing to 1. It takes a table of positions too, which Model
1 does not have; Step 3 supplies one.


In [ ]:
def start(pairs):
    """Round 0: every French word equally likely for every English word."""
    english = {e for en, _ in pairs for e in en}
    french = {f for _, fr in pairs for f in fr}
    return {(f, e): 1 / len(french) for e in english for f in french}


def model1_round(t, pairs):
    """One round of Model 1's EM, as in Lecture 18's notebook."""
    count, total = defaultdict(float), defaultdict(float)
    for english, french in pairs:
        for f in french:
            z = sum(t[f, e] for e in english)
            for e in english:
                count[f, e] += t[f, e] / z
                total[e] += t[f, e] / z
    return {(f, e): count[f, e] / total[e] for (f, e) in t}


def alignment(t, english, french, a=None):
    """Each French word's share across the English words; with a, position counts too."""
    l, m = len(english), len(french)
    rows = []
    for j, f in enumerate(french):
        weight = [t[f, e] * (a[i, j, l, m] if a else 1) for i, e in enumerate(english)]
        rows.append([w / sum(weight) for w in weight])
    return torch.tensor(rows)


def show(t, english, french, a=None):
    english, french = english.split(), french.split()
    print(format_attention(alignment(t, english, french, a), english, french, show_values=True))


t1 = start(pairs)
for _ in range(10):
    t1 = model1_round(t1, pairs)

show(t1, "the cat sees the dog", "le chat voit le chien")


Look at the two `le` rows: they are the same. Model 1 has learned that `le` goes with `the`, but
this sentence has two of each, and a model that never looks at positions cannot say which `le`
belongs to which `the`, though the first `le` plainly translates the first `the`. Move the words
of either sentence around and the shares stay where they are, moved with their words.

Now the two hard pairs:


In [ ]:
show(t1, "the young fox sees the cat", "le jeune renard voit le chat")
print()
show(t1, "the grey owl sees the cat", "la chouette grise voit le chat")


Fifty-fifty, both times. `jeune` and `renard` appear together with `young` and `fox` once, and
nowhere else; nothing in the counts favours either. The same holds for the owl. Model 1 cannot
do better, and it should not pretend to.

## Step 3: Model 2 adds a table of positions

Model 2 keeps t, and adds a second table, a(i | j, l, m): in a pair whose English sentence has
`l` words and whose French sentence has `m`, the probability that the French word at position `j`
translates the English word at position `i`. The same EM learns both. In the expect step each
French word's credit is shared in proportion to **t × a**, what the words say times what the
positions say; in the maximize step the shares are added up twice, once by word into a new t and
once by position into a new a.

The cell below is that round, written out. Following Brown and colleagues, Model 2 starts from
Model 1's t, with every position equally likely, and runs ten rounds of its own.


In [ ]:
def uniform_positions(pairs):
    """Round 0 of a: every English position equally likely, for each pair of lengths."""
    a = {}
    for english, french in pairs:
        l, m = len(english), len(french)
        for j in range(m):
            for i in range(l):
                a[i, j, l, m] = 1 / l
    return a


def model2_round(t, a, pairs):
    """One round of Model 2's EM: share by t times a, then renormalize both tables."""
    count, total = defaultdict(float), defaultdict(float)   # by word, for t
    at, at_total = defaultdict(float), defaultdict(float)   # by position, for a
    for english, french in pairs:
        l, m = len(english), len(french)
        for j, f in enumerate(french):
            # Expect: each English word gets a share of f, by what the words and positions say.
            z = sum(t[f, e] * a[i, j, l, m] for i, e in enumerate(english))
            for i, e in enumerate(english):
                share = t[f, e] * a[i, j, l, m] / z
                count[f, e] += share
                total[e] += share
                at[i, j, l, m] += share
                at_total[j, l, m] += share
    # Maximize: a new t from the shares by word, a new a from the shares by position.
    t = {(f, e): count[f, e] / total[e] for (f, e) in t}
    a = {(i, j, l, m): at[i, j, l, m] / at_total[j, l, m] for (i, j, l, m) in a}
    return t, a


def model2(pairs, rounds=10):
    """Model 1 for ten rounds, then Model 2 for `rounds`; returns both tables."""
    t = start(pairs)
    for _ in range(10):
        t = model1_round(t, pairs)
    a = uniform_positions(pairs)
    for _ in range(rounds):
        t, a = model2_round(t, a, pairs)
    return t, a


t2, a2 = model2(pairs)


def show_positions(a, l, m):
    """The table of positions for one pair of lengths: a row per French position."""
    table = torch.tensor([[a[i, j, l, m] for i in range(l)] for j in range(m)])
    english = [f"e{i + 1}" for i in range(l)]
    french = [f"f{j + 1}" for j in range(m)]
    print(format_attention(table, english, french, show_values=True))


print("What Model 2 learned about positions in six-word pairs:")
show_positions(a2, 6, 6)
print()
print("and in four-word pairs:")
show_positions(a2, 4, 4)


**Six-word pairs: a diagonal.** French word 1 translates English word 1, word 2 word 2, and so
on. Of the five six-word pairs, four keep English order, and the counts of the words were clear
enough in them to teach the diagonal.

**Four-word pairs: words 2 and 3 swap.** Both four-word pairs in the corpus have an adjective
after its noun, *le chat noir dort* and *un chien blanc court*, so for this length Model 2 has
learned that French word 2 translates English word 3. Its positions are absolute, one table for
each pair of sentence lengths, and each table learns only from the pairs of its own lengths.

With the diagonal, the first problem of Step 2 goes away:


In [ ]:
show(t2, "the cat sees the dog", "le chat voit le chien", a2)
print()
show(t2, "the young fox sees the cat", "le jeune renard voit le chat", a2)


Each `le` now goes to its own `the`, and the fox is settled, `jeune` on `young` and `renard` on
`fox`. The words could not decide between them, so the positions did, and the positions were
right.

## Step 4: predict, then align the owl

The owl's pair is six words long too, and its words are as undecided as the fox's were: Model 1
split `chouette` and `grise` fifty-fifty between `grey` and `owl`. **Write a prediction first**,
in `PREDICTION` below: will Model 2 align `chouette` and `grise` correctly, and why? The tables
of Steps 2 and 3 hold the answer. The notebook stops at the next cell until you write it.


In [ ]:
PREDICTION = ""   # <- your prediction, in a sentence or two
assert PREDICTION.strip(), "Write your prediction in PREDICTION, then run this cell again."


In [ ]:
OWL = ("the grey owl sees the cat", "la chouette grise voit le chat")

print("Model 2, the words times the positions:")
show(t2, *OWL, a2)
print()
print("What Model 2 now believes about the words, its t alone:")
show(t2, *OWL)
print()
print("You predicted:", PREDICTION.strip())


Model 2 is sure, and wrong: `chouette` (owl) on `grey`, and `grise` (grey) on `owl`. In its first
round the words were undecided, fifty-fifty as Model 1 left them, so the positions decided: the
diagonal sent each French word to the English word in the same place. The maximize step then
counted those shares into t, and the second table shows the result. Model 2 now believes that
*chouette* means *grey*, a wrong dictionary entry learned from position alone.

The same reasoning was right for the fox, whose French keeps English order, and is wrong for
the owl, whose French reverses it. Model 2 learned its diagonal from pairs that keep the order,
and nothing in it can know that *grise* is the kind of adjective that follows its noun.

## Step 5: more evidence from the words

The fix one might expect is more data: pairs in which `owl` appears without `grey`, so that the
counts can tell `chouette` from `grise`. The cell adds two and trains both models again.


In [ ]:
MORE_OWLS = [("the owl sleeps", "la chouette dort"), ("an owl runs", "une chouette court")]
more_pairs = pairs + [(english.split(), french.split()) for english, french in MORE_OWLS]

t1_more = start(more_pairs)
for _ in range(10):
    t1_more = model1_round(t1_more, more_pairs)
t2_more, a2_more = model2(more_pairs)

print("Model 1, with two more owls:")
show(t1_more, *OWL)
print()
print("Model 2, with two more owls:")
show(t2_more, *OWL, a2_more)
print()
print(f"Model 2's own t: chouette for owl {t2_more['chouette', 'owl']:.2f}, "
      f"for grey {t2_more['chouette', 'grey']:.2f}")


The words now point the right way: Model 1 puts `chouette` on `owl`, and `grise` follows to
`grey`. Model 2 still puts `chouette` on `grey`, though its own t has learned that *chouette*
means *owl*. Over its ten rounds the table of positions for six-word pairs became certain, 100
on the diagonal and 0 off it, learned from the four pairs that keep English order. A share goes
in proportion to t × a, so a position near 0 cancels a word, however sure the word is. On a
corpus this small, four sentences' positions overrule the words.

Real aligners soften this. **fast_align** (Dyer, Chahuneau, and Smith, NAACL 2013), a
reparametrization of Model 2, replaces the table for each pair of lengths with one learned
number: how strongly to prefer the diagonal, measured by relative position, so that sentences of
every length share it. It still prefers the diagonal, and so still meets trouble in languages
that reorder.

## Step 6: how attention uses position

A Transformer's cross-attention gives the same kind of row: for each French word it writes, a
weight on each English word, the weights summing to 1. But it keeps no table of positions. Each
weight is computed from the words, by comparing the decoder's state as it writes `chouette`
with each English word's vector, and that state is built from the words written so far, here
`la`. Position is in those vectors too, added to each word at the input as a positional
encoding (Lecture 8c), so a Transformer can learn that French adjectives often follow their
nouns. But position is one part of what each vector says, weighed together with the word itself
and its neighbours, not a separate table multiplied in, and the same for every sentence length.
So when the words are clear, the position need not overrule them, and when the order changes,
the content can follow it.

Nor is attention trained to align, as Lecture 18's notebook showed: it spreads its weight where
reading helps predict the next word. Its rows are the closest thing a Transformer has to Model
2's alignment, and [tutorial 4](04-attention-and-alignment.ipynb) measures how closely they
match the right one.

**For your Final Project.** If you align words, to check that a required term was translated,
or to see which source words an error came from, the aligner you choose trusts some evidence
more than other. fast_align trusts the words and a preference for the diagonal, which suits
languages with similar order. awesome-align (Dou and Neubig, EACL 2021, a Lecture 18 reading)
compares the words' vectors in a multilingual model, so it relies on content, and on how well
that model knows your language.

## Your turn: a four-word pair in English order

Step 3's four-word table learned a swap, because both four-word pairs had one. Add
`("the small dog sleeps", "le petit chien dort")` to `PAIRS` in Step 1, where *petit* comes
before the noun as in English. Before you run anything, decide what the four-word table will do,
and whether Model 2 will align the new pair correctly. Then run the `PAIRS` cell and, with it
still selected, choose **Runtime → Run after**.
